# MNIST CNN Mini-Project
Handwritten digits: 28x28 grayscale images, 10 classes (0-9).

Steps: load and inspect, split, build a CNN, check feature-map sizes, train, evaluate,
error analysis, and look inside the network (filters and feature maps).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras import Input, Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Rescaling, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
)

SEED = 42
EPOCHS = 20          # upper limit; EarlyStopping usually stops sooner
BATCH_SIZE = 128
tf.keras.utils.set_random_seed(SEED)

# Where is your data?
#   "keras": built-in MNIST (downloads automatically, needs internet the first time)
#   "csv"  : a Kaggle-style CSV with a "label" column followed by 784 pixel columns
DATA_SOURCE = "keras"
CSV_PATH = "mnist_train.csv"

CLASS_NAMES = [str(i) for i in range(10)]

## 1. Load and inspect the data
Images stay as raw `uint8` (0-255); scaling to 0-1 happens inside the model.

In [ ]:
if DATA_SOURCE == "keras":
    (X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
else:
    df = pd.read_csv(CSV_PATH)
    y_all = df["label"].values
    X_all = df.drop(columns="label").values.reshape(-1, 28, 28).astype("uint8")
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X_all, y_all, test_size=0.15, random_state=SEED, stratify=y_all,
    )

# CNNs expect a channel axis: (28, 28) -> (28, 28, 1)
X_train_full = X_train_full[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("X_train:", X_train_full.shape, X_train_full.dtype)
print("X_test: ", X_test.shape)
print("pixel range:", X_train_full.min(), "to", X_train_full.max())

In [ ]:
def show_images(images, titles, rows, cols, figsize=(10, 6), cmap=None):
    """Display a grid of images with titles."""
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if i >= rows * cols:
            break
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img.squeeze(), cmap=cmap)
        plt.title(title, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


show_images(X_train_full[:12], [f"Label: {y}" for y in y_train_full[:12]],
            rows=3, cols=4, figsize=(8, 6), cmap="gray")

# Class balance
pd.Series(y_train_full).value_counts().sort_index().plot(kind="bar", title="Images per digit")
plt.show()

## 2. Train / validation split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.1, random_state=SEED, stratify=y_train_full,
)
print("train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)

## 3. Build the CNN

In [ ]:
def build_model(dropout=0.5):
    model = Sequential(name="mnist_cnn")
    model.add(Input(shape=(28, 28, 1)))
    model.add(Rescaling(1.0 / 255))
    model.add(Conv2D(32, (3, 3), activation="relu"))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation="relu"))
    model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation="relu"))
    model.add(Dropout(dropout))
    model.add(Dense(10, activation="softmax"))
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_model()
model.summary()   # total parameters should be 225,034

### Board exercise: feature-map sizes

| Layer | Output shape |
|---|---|
| Input | 28 x 28 x 1 |
| Conv2D(32, 3x3) | 26 x 26 x 32 |
| MaxPooling | 13 x 13 x 32 |
| Conv2D(64, 3x3) | 11 x 11 x 64 |
| MaxPooling | 5 x 5 x 64 |
| Flatten | 1600 |

Parameter check for the first conv layer: 3 x 3 x 1 x 32 weights + 32 biases = 320.

## 4. Train

In [ ]:
def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(history.history[metric], label="train")
        ax.plot(history.history["val_" + metric], label="validation")
        ax.set_xlabel("epoch")
        ax.set_title(f"{title} {metric}".strip())
        ax.legend()
    plt.tight_layout()
    plt.show()


history = model.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    verbose=2,
)
plot_history(history, "MNIST")

## 5. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  |  test loss: {test_loss:.4f}")

probs = model.predict(X_test, verbose=0)
y_pred = probs.argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

## 6. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title("Counts")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set_title("Normalized by actual class")
for ax in axes:
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
print("Most confused pairs:")
for a, p in np.dstack(np.unravel_index(np.argsort(-off_diag, axis=None), off_diag.shape))[0][:5]:
    print(f"  digit {a} predicted as {p}: {off_diag[a, p]}")

## 7. Error analysis

In [ ]:
wrong = np.where(y_test != y_pred)[0]
print(f"{len(wrong)} wrong predictions out of {len(y_test)}")

show_images(
    X_test[wrong[:12]],
    [f"True: {y_test[i]}  Pred: {y_pred[i]}" for i in wrong[:12]],
    rows=3, cols=4, figsize=(9, 7), cmap="gray",
)

In [ ]:
# Confidently wrong: the model was sure, and still wrong
confidence = probs.max(axis=1)
confident_wrong = wrong[np.argsort(-confidence[wrong])][:12]

show_images(
    X_test[confident_wrong],
    [f"True: {y_test[i]}  Pred: {y_pred[i]} ({confidence[i]:.0%})" for i in confident_wrong],
    rows=3, cols=4, figsize=(9, 7), cmap="gray",
)

## 8. Look inside the network
**Filters** are the small 3x3 patterns the first layer learned.
**Feature maps** show how those filters respond to one specific digit.

In [ ]:
conv_layers = [l for l in model.layers if isinstance(l, Conv2D)]

filters = conv_layers[0].get_weights()[0]          # shape (3, 3, 1, 32)
plt.figure(figsize=(10, 3))
for i in range(filters.shape[-1]):
    plt.subplot(4, 8, i + 1)
    plt.imshow(filters[:, :, 0, i], cmap="gray")
    plt.axis("off")
plt.suptitle("Layer 1 filters (3x3)")
plt.show()

In [ ]:
fm_model = Model(inputs=model.inputs, outputs=[l.output for l in conv_layers])
sample_idx = 0
maps = fm_model.predict(X_test[sample_idx:sample_idx + 1], verbose=0)

plt.figure(figsize=(3, 3))
plt.imshow(X_test[sample_idx].squeeze(), cmap="gray")
plt.title(f"Input (label {y_test[sample_idx]})")
plt.axis("off")
plt.show()

for layer_no, fmap in enumerate(maps, start=1):
    n_show = 8
    plt.figure(figsize=(12, 2))
    for i in range(n_show):
        plt.subplot(1, n_show, i + 1)
        plt.imshow(fmap[0, :, :, i], cmap="viridis")
        plt.axis("off")
    plt.suptitle(f"Conv layer {layer_no}: first {n_show} feature maps, size {fmap.shape[1]}x{fmap.shape[2]}")
    plt.show()

## 9. Save the model

In [ ]:
model.save("mnist_cnn.keras")
print("saved to mnist_cnn.keras")

## Discussion questions
1. Which digits are confused most often, and why might they look alike when handwritten?
2. Look at the confidently wrong images. Would a person also find some of them ambiguous?
3. The feature maps shrink from 26x26 to 11x11. What information is kept and what is lost?
4. MNIST reaches very high accuracy with a small CNN, yet CIFAR-10 does not. What makes CIFAR-10 harder?